In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from enum import Enum

import typing as t

# Data types for columns
class DataType(Enum):
    DATE = 0
    DATETIME = 1
    TEXT = 2
    FLOAT = 3
    INT = 4

# Years to process
FIRST_YEAR = 2017
LAST_YEAR = 2022

Common functions

In [75]:
# Custom process
def check_is_header(series: pd.Series) -> bool:
    return series.str.match(r"^[A-Za-z_]+$").all()

def default_custom_process(df: pd.DataFrame) -> pd.DataFrame:
    is_header = check_is_header(df.iloc[0])
    if is_header:
        df = df.iloc[1:, :].reset_index(drop=True)
        
    return df

# Process paths
def get_unique_files(file_list:t.List[Path]) -> t.List[Path]:
    unique_files = {}
    for file in file_list:
        name_without_extension = "".join(file.name.split(".")[:-1])
        date = name_without_extension.split("_")[-1]  # Assuming the date is the last part of the name
        if file.name not in unique_files:
            unique_files[date] = file
    return list(unique_files.values())

# Process datetimes
def prepare_datetime(time_str: str) -> str:
    """ 
    Possible values for time_str: [
        "01/01/3000 9:00:06", 
        "01/01/3000 09:00:06", 
        "09:00:00.000000", 
        "09:00:00:000", 
        "08:00:16.002360"
    ]
    """
    time_str = time_str.split(" ")[-1]
    time_str_split = time_str.split(":")
    time_str_split[0] = f"0{time_str_split[0]}" if len(time_str_split[0]) == 1 else time_str_split[0]

    if len(time_str_split) == 4:
        time_str = f"{time_str_split[0]}:{time_str_split[1]}:{time_str_split[2]}.{time_str_split[3]}"
    elif len(time_str_split) != 3:
        raise ValueError(f"Unexpected time format: {time_str}")
    else:
        time_str = ":".join(time_str_split)

    time_str_split = time_str.split(".")
    if len(time_str_split) == 1:
        time_str = f"{time_str_split[0]}.000000"

    return time_str

# Main function to build raw data
def build_data_raw(
        columns_list:t.List[str],
        selected_columns_dict:t.Dict[str, DataType],
        file_prefix: str,
        custom_processing_func: t.Optional[t.Callable] = default_custom_process
    ) -> pd.DataFrame:

    # Data raw
    data_raw = []

    # For every year, we read each contract type
    for year in tqdm(range(FIRST_YEAR, LAST_YEAR + 1)):
        path_year = Path(f"data/{year}")

        file_list = list(path_year.glob(f"{file_prefix}_*.TXT")) + list(path_year.glob(f"{file_prefix}_*.M3"))
        unique_file_list = get_unique_files(file_list)

        for file in unique_file_list:
            if file.is_file():
                df = pd.read_csv(
                    file,
                    delimiter=";",
                    header=None,
                    dtype="string",
                )

                # Custom process for each case. We can use the default one, which checks if the first row is a header and removes it if so.
                df = custom_processing_func(df=df)

                # Columns
                total_columns = df.shape[1]
                unknown_names = [
                    f"unknown_{i+1}"
                    for i in range(max(0, total_columns - len(columns_list)))
                ]
                column_names = columns_list + unknown_names

                # Assign
                df.columns = column_names

                # Select only relevant columns
                df = df[list(selected_columns_dict.keys())]

                # IBX mask
                IBX_mask = df["ContractCode"].str.contains(
                    ("IBX"),
                    na=False
                )
                df = df[IBX_mask]

                # Convert data types
                for col, dtype in selected_columns_dict.items():
                    if dtype == DataType.DATE:
                        df[col] = pd.to_datetime(df[col], format="%Y%m%d").dt.date    
                    elif dtype == DataType.DATETIME:
                        
                        try:
                            df[col] = df[col].str.strip().str.split(" ").str[-1].apply(prepare_datetime)
                            df[col] = pd.to_datetime(df[col], format="%H:%M:%S.%f")
                        except Exception as e:
                            raise ValueError(f"Unexpected format in column {col}. file: {file.name} Error: {e}")
                    elif dtype == DataType.FLOAT:
                        df[col] = pd.to_numeric(df[col].str.replace(",", "."), downcast="float")
                    elif dtype == DataType.INT:
                        df[col] = pd.to_numeric(df[col], downcast="integer")
                
                # Metadata
                df["Year"] = year
                df["SourceFile"] = file.name
                
                # Join data_raw
                data_raw.append(df)

    # Concatenate final DataFrame
    data_raw = pd.concat(data_raw, ignore_index=True)

    # Save CSV
    output_dir = Path(f"raw_data")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{file_prefix}.csv"
    data_raw.to_csv(output_file, index=False, encoding="utf-8")

    print(f"\nArchivo guardado en: {output_file}")
    print(f"Total filas finales: {len(data_raw)}")

    return data_raw

CCONTRACTS_C2

In [18]:
CCONTRACTS_C2_COLUMNS_LIST = [
    "SessionDate",                     # 1 Fecha de sesión
    "ClearingHouseCode",               # 2 Código de cámara
    "ContractCode",                    # 3 Código de contrato
    "ContractGroupCode",               # 4 Grupo del contrato
    "ContractTypeCode",                # 5 Tipo del contrato
    "StrikePrice",                     # 6 Precio de ejercicio
    "MaturityDate",                    # 7 Fecha de vencimiento
    "TradingEndDate",                  # 8 Fecha de fin de negociación
    "ExerciseUnderlyingContractCode",  # 9 Código contrato subyacente (ejercicio)
    "MarginUnderlyingContractCode",    # 10 Código contrato subyacente (garantías)
    "ArrayCode",                       # 11 Código de matriz de garantías
    "ExpiryNumber",                    # 12 Nº vencimiento de liquidación
    "OffsetNumber",                    # 13 Nº compensación
    "ExpirySpan",                      # 14 Tipo de vencimiento (S/L)
    "MaturityMonthYear",               # 15 Identificador del vencimiento
    "ISINCode"                         # 16 Código ISIN
]

CCONTRACTS_C2_COLUMNS_SELECTED_DICT = {
    "SessionDate": DataType.DATE,       # 1 Fecha de sesión
    "ContractCode": DataType.TEXT,      # 3 Código de contrato
    "StrikePrice": DataType.FLOAT,      # 6 Precio de ejercicio
    "MaturityDate": DataType.DATE,      # 7 Fecha de vencimiento
}

In [19]:
# CCONTRACTS_C2
df = build_data_raw(
    columns_list=CCONTRACTS_C2_COLUMNS_LIST,
    selected_columns_dict=CCONTRACTS_C2_COLUMNS_SELECTED_DICT,
    file_prefix="CCONTRACTS_C2",
    )


100%|██████████| 6/6 [01:58<00:00, 19.69s/it]



Archivo guardado en: raw_data\CCONTRACTS_C2.csv
Total filas finales: 2520238


In [15]:
# NAs in StrikePrice are expected, as there are some contracts (like futures) that do not have a strike price.
df.isna().sum()

SessionDate         0
ContractCode        0
StrikePrice     22916
MaturityDate        0
Year                0
SourceFile          0
dtype: int64

In [20]:
df

,SessionDate,ContractCode,StrikePrice,MaturityDate,Year,SourceFile
0,2017-03-17,FIBXM7,<NA>,2017-06-16,2017,CCONTRACTS_C2_20170317.TXT
1,2017-03-17,FIBXZ7,<NA>,2017-12-15,2017,CCONTRACTS_C2_20170317.TXT
2,2017-03-17,CIBX 8000M17,8000.0,2017-06-16,2017,CCONTRACTS_C2_20170317.TXT
3,2017-03-17,CIBX 7800Z17,7800.0,2017-12-15,2017,CCONTRACTS_C2_20170317.TXT
4,2017-03-17,CIBX 7900Z17,7900.0,2017-12-15,2017,CCONTRACTS_C2_20170317.TXT
...,...,...,...,...,...,...
2520233,2022-06-28,PIBX 9200W4N22,9200.0,2022-07-22,2022,CCONTRACTS_C2_20220628.TXT
2520234,2022-06-28,PIBX 9300W4N22,9300.0,2022-07-22,2022,CCONTRACTS_C2_20220628.TXT
2520235,2022-06-28,PIBX 9400W4N22,9400.0,2022-07-22,2022,CCONTRACTS_C2_20220628.TXT
2520236,2022-06-28,CIBX 9500W4N22,9500.0,2022-07-22,2022,CCONTRACTS_C2_20220628.TXT


TGENTRADES

In [34]:
TGENTRADES_COLUMNS_LIST = [
    "SessionDate",    # 1 Fecha de sesión
    "MarketCode",     # 2 Código de mercado
    "TradeExecID",    # 3 Número de registro de negociación
    "ContractCode",   # 4 Código de contrato
    "ExecTime",       # 5 Hora de ejecución
    "TradePrice",     # 6 Precio
    "Quantity",       # 7 Volumen
    "TradeType"       # 8 Tipo de operación
]

TGENTRADES_COLUMNS_SELECTED_DICT = {
    "SessionDate": DataType.DATE,       # 1 Fecha de sesión
    "MarketCode": DataType.TEXT,        # 2 Código de mercado
    "TradeExecID": DataType.TEXT,       # 3 Número de registro de negociación
    "ContractCode": DataType.TEXT,      # 4 Código de contrato
    "ExecTime": DataType.DATETIME,      # 5 Hora de ejecución
    "TradePrice": DataType.FLOAT,       # 6 Precio
    "Quantity": DataType.INT,           # 7 Volumen
    "TradeType": DataType.TEXT          # 8 Tipo de operación
}

In [53]:
def tgentrades_custom_process(df: pd.DataFrame) -> pd.DataFrame:
    is_header = check_is_header(df.iloc[0])
    if is_header:
        skip_column_list = [c for c, v in df.iloc[0].items() if v.lower().strip() == "secuencia"]
        df.drop(columns=skip_column_list, inplace=True)
        df = df.iloc[1:, :].reset_index(drop=True)

    return df

In [ ]:
def prepare_datetime(time_str: str) -> str:
    """ 
    Possible values for time_str: [
        "01/01/3000 9:00:06", 
        "01/01/3000 09:00:06", 
        "09:00:00.000000", 
        "09:00:00:000", 
        "08:00:16.002360"
    ]
    """
    time_str = time_str.split(" ")[-1]
    time_str_split = time_str.split(":")
    time_str_split[0] = f"0{time_str_split[0]}" if len(time_str_split[0]) == 1 else time_str_split[0]

    if len(time_str_split) == 4:
        time_str = f"{time_str_split[0]}:{time_str_split[1]}:{time_str_split[2]}.{time_str_split[3]}"
    elif len(time_str_split) != 3:
        raise ValueError(f"Unexpected time format: {time_str}")
    else:
        time_str = ":".join(time_str_split)

    time_str_split = time_str.split(".")
    if len(time_str_split) == 1:
        time_str = f"{time_str_split[0]}.000000"

    return time_str

series = pd.Series(["01/01/3000 9:00:06", "01/01/3000 09:00:06", "09:00:00.000000", "09:00:00:000", "08:00:16.002360"])

series = series.str.split(" ").str[-1].apply(prepare_datetime)

display(series)
pd.to_datetime(series, format="%H:%M:%S.%f")

0    09:00:06.000000
1    09:00:06.000000
2    09:00:00.000000
3       09:00:00.000
4    08:00:16.002360
dtype: object

0   1900-01-01 09:00:06.000000
1   1900-01-01 09:00:06.000000
2   1900-01-01 09:00:00.000000
3   1900-01-01 09:00:00.000000
4   1900-01-01 08:00:16.002360
dtype: datetime64[ns]

In [76]:
# TGENTRADES
df = build_data_raw(
    columns_list=TGENTRADES_COLUMNS_LIST,
    selected_columns_dict=TGENTRADES_COLUMNS_SELECTED_DICT,
    file_prefix="TGENTRADES",
    custom_processing_func=tgentrades_custom_process,
    )

100%|██████████| 6/6 [01:48<00:00, 18.12s/it]



Archivo guardado en: raw_data\TGENTRADES.csv
Total filas finales: 17197910


In [14]:
df.isna().sum()

SessionDate     0
MarketCode      0
TradeExecID     0
ContractCode    0
ExecTime        0
TradePrice      0
Quantity        0
TradeType       0
Year            0
SourceFile      0
dtype: int64

In [77]:
df

,SessionDate,MarketCode,TradeExecID,ContractCode,ExecTime,TradePrice,Quantity,TradeType,Year,SourceFile
0,2017-01-02,M3,FI0050130697,FIBXF7,1900-01-01 09:00:06.000000,9296.0,6,M,2017,TGENTRADES_M3_20170102.TXT
1,2017-01-02,M3,FI0050130698,FIBXF7,1900-01-01 09:00:06.000000,9296.0,1,M,2017,TGENTRADES_M3_20170102.TXT
2,2017-01-02,M3,FI0050130699,FIBXF7,1900-01-01 09:00:06.000000,9296.0,1,M,2017,TGENTRADES_M3_20170102.TXT
3,2017-01-02,M3,FI0050130700,FIBXF7,1900-01-01 09:00:06.000000,9296.0,1,M,2017,TGENTRADES_M3_20170102.TXT
4,2017-01-02,M3,FI0050130701,FIBXF7,1900-01-01 09:00:06.000000,9296.0,1,M,2017,TGENTRADES_M3_20170102.TXT
...,...,...,...,...,...,...,...,...,...,...
17197905,2022-06-28,M3,007647891,FIBXN2,1900-01-01 19:59:28.277066,8188.0,1,M,2022,TGENTRADES_20220628.M3
17197906,2022-06-28,M3,007647892,FIBXN2,1900-01-01 19:59:44.393052,8188.0,1,M,2022,TGENTRADES_20220628.M3
17197907,2022-06-28,M3,007647893,FIBXN2,1900-01-01 19:59:47.739983,8186.0,1,M,2022,TGENTRADES_20220628.M3
17197908,2022-06-28,M3,007647894,FIBXN2,1900-01-01 19:59:54.692627,8190.0,1,M,2022,TGENTRADES_20220628.M3


CCONTRSTAT

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

first_year = 2017
last_year = 2022


# CCONTRSTAT_C2 columns
CCONTRSTAT_C2_columns = [
    "SessionDate",                  # 1 Fecha de sesión
    "ClearingHouseCode",            # 2 Código de cámara
    "ContractCode",                 # 3 Código de contrato
    "HighPrice",                    # 4 Precio alto
    "LowPrice",                     # 5 Precio bajo
    "FirstPrice",                   # 6 Primer precio
    "LastPrice",                    # 7 Último precio
    "SettlPrice",                   # 8 Precio de liquidación
    "SettlVolatility",              # 9 Volatilidad de liquidación
    "SettlDelta",                   # 10 Delta de liquidación
    "PreviousDaySettlPrice",        # 11 Precio liquidación sesión anterior
    "PreviousDaySettlVolatility",   # 12 Volatilidad sesión anterior
    "PreviousDaySettlDelta",        # 13 Delta sesión anterior
    "TotalRegVolume",               # 14 Volumen total registrado
    "NumberOfTrades",               # 15 Número de operaciones
    "OpenInterest"                  # 16 Posición abierta
]


# Raw data
CCONTRSTAT_C2_raw = []

# For every year, we read each contract type
for year in tqdm(range(first_year, last_year + 1)):
    path_year = Path(f"data/{year}")
    
    # Process monitoring
    total_archives_CCONTRSTAT_C2_year = 0

    ### CCONTRSTAT_C2 ###
    for archive_CCONTRSTAT_C2 in path_year.glob("CCONTRSTAT_C2_*.TXT"):
        if archive_CCONTRSTAT_C2.is_file():
            # Read archive
            df_aux_CCONTRSTAT_C2 = pd.read_csv(
                archive_CCONTRSTAT_C2,
                delimiter=";",
                header=None,
                dtype=str,
                low_memory=False
            )

            # Columns
            total_columns = df_aux_CCONTRSTAT_C2.shape[1]
            unknown_names = [
                f"unknown_{i+1}"
                for i in range(max(0, total_columns - len(CCONTRSTAT_C2_columns)))
            ]
            column_names = CCONTRSTAT_C2_columns + unknown_names
            # Assign
            df_aux_CCONTRSTAT_C2.columns = column_names

            # IBX_mask
            IBX_mask = df_aux_CCONTRSTAT_C2["ContractCode"].str.contains(
                ("IBX"),
                na=False
            )

            df_aux_CCONTRSTAT_C2 = df_aux_CCONTRSTAT_C2[IBX_mask]

            # Metadata
            df_aux_CCONTRSTAT_C2["Year"] = year
            df_aux_CCONTRSTAT_C2["SourceFile"] = archive_CCONTRSTAT_C2.name
            
            # Join CCONTRSTAT_C2_raw
            CCONTRSTAT_C2_raw.append(df_aux_CCONTRSTAT_C2)

            # Count of this type of archive in the year
            total_archives_CCONTRSTAT_C2_year += 1
    
    print(f"Total archives CCONTRSTAT_C2_* year {year} -> {total_archives_CCONTRSTAT_C2_year}")

# Concatenate final DataFrame
CCONTRSTAT_C2_raw = pd.concat(CCONTRSTAT_C2_raw, ignore_index=True)

# Order columns
unknown_cols = sorted(
    [c for c in CCONTRSTAT_C2_raw.columns if c.startswith("unknown_")],
    key=lambda x: int(x.split("_")[1])
)
final_columns_order = (
    CCONTRSTAT_C2_columns +
    unknown_cols +
    ["Year", "SourceFile"]
)
CCONTRSTAT_C2_raw = CCONTRSTAT_C2_raw.reindex(columns=final_columns_order)

# Save CSV
output_dir = Path(f"raw_data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "CCONTRSTAT_C2_raw.csv"
CCONTRSTAT_C2_raw.to_csv(output_file, index=False, encoding="utf-8")

print(f"\nArchivo guardado en: {output_file}")
print(f"Total filas finales: {len(CCONTRSTAT_C2_raw)}")

# View
display(CCONTRSTAT_C2_raw)

# Exploratoring
CCONTRSTAT_C2_raw.isna().sum()

 17%|█▋        | 1/6 [00:12<01:00, 12.17s/it]

Total archives CCONTRSTAT_C2_* year 2017 -> 200


 33%|███▎      | 2/6 [00:29<01:00, 15.17s/it]

Total archives CCONTRSTAT_C2_* year 2018 -> 255


 33%|███▎      | 2/6 [00:40<01:20, 20.23s/it]


KeyboardInterrupt: 

Las columnas que después vamos a tomar son las siguientes:
* SessionDate
* ContractCode
* HighPrice
* LowPrice
* FirstPrice
* LastPrice

Para estas, las columnas de precios presentan el mismo númeor de valores nulos, habrá que identificar qué ha ocurrido.

# Construcción TRADE_IBEX_DATABASE

In [ ]:
import pandas as pd
from pathlib import Path

# Read CSVs
# CCONTRACTS_C2
# Columns
CCONTRACTS_C2_columns_merge = [
    "SessionDate",                     # 1 Fecha de sesión
    "ContractCode",                    # 3 Código de contrato
    "StrikePrice",                     # 6 Precio de ejercicio
    "MaturityDate",                    # 7 Fecha de vencimiento
]

contracts_file = Path("raw_data/CCONTRACTS_C2_raw.csv")
CCONTRACTS_C2_raw = pd.read_csv(contracts_file, delimiter=",", dtype=str, encoding="utf-8")
CCONTRACTS_C2_raw = CCONTRACTS_C2_raw.loc[:,CCONTRACTS_C2_columns_merge]

# CCONTRSTAT_C2
# Columns
CCONTRSTAT_C2_columns_merge = [
    "SessionDate",                  # 1 Fecha de sesión
    "ContractCode",                 # 3 Código de contrato
    "HighPrice",                    # 4 Precio alto
    "LowPrice",                     # 5 Precio bajo
    "FirstPrice",                   # 6 Primer precio
    "LastPrice",                    # 7 Último precio
]

contrstat_file = Path("raw_data/CCONTRSTAT_C2_raw.csv")
CCONTRSTAT_C2_raw = pd.read_csv(contrstat_file, delimiter=",", dtype=str, encoding="utf-8")
CCONTRSTAT_C2_raw = CCONTRSTAT_C2_raw.loc[:,CCONTRSTAT_C2_columns_merge]

# TGENTRADES
# Columns
TGENTRADES_columns_merge = [
    "SessionDate",    # 1 Fecha de sesión
    "MarketCode",     # 2 Código de mercado
    "TradeExecID",    # 3 Número de registro de negociación
    "ContractCode",   # 4 Código de contrato
    "ExecTime",       # 5 Hora de ejecución
    "TradePrice",     # 6 Precio
    "Quantity",       # 7 Volumen
    "TradeType"       # 8 Tipo de operación
]

tgentrades_file = Path("raw_data/TGENTRADES_raw.csv")
TGENTRADES_raw = pd.read_csv(tgentrades_file, delimiter=",", dtype=str, encoding="utf-8")
TGENTRADES_raw = TGENTRADES_raw.loc[:,TGENTRADES_columns_merge]

# Add contract information
TRADE_IBEX_DATABASE = TGENTRADES_raw.merge(
    CCONTRACTS_C2_raw,
    on=["SessionDate","ContractCode"],
    how="left",   # left join mantiene toda la info de TGENTRADES
    suffixes=("", "_contract")  # si hay columnas repetidas
)

# Add contract stats information
TRADE_IBEX_DATABASE = TRADE_IBEX_DATABASE.merge(
    CCONTRSTAT_C2_raw,
    on=["SessionDate","ContractCode"],
    how="left",
    suffixes=("", "_stat")
)

# Save CSV
output_file = Path("raw_data/TRADE_IBEX_DATABASE.csv")
TRADE_IBEX_DATABASE.to_csv(output_file, index=False, encoding="utf-8")
print(f"Archivo final guardado en: {output_file}")

display(TRADE_IBEX_DATABASE)

Archivo final guardado en: raw_data\TRADE_IBEX_DATABASE.csv


,SessionDate,MarketCode,TradeExecID,ContractCode,ExecTime,TradePrice,Quantity,TradeType,StrikePrice,MaturityDate,HighPrice,LowPrice,FirstPrice,LastPrice
0,20170317,M3,FI0050674468,FIBXH7,08:00:26,10166,1,M,NaN,20170317,10226,10142,10166,10216
1,20170317,M3,FI0050674469,FIBXJ7,08:00:26,10129,1,M,NaN,20170421,10231,10101,10129,10224
2,20170317,M3,FI0050674470,FIBXH7,08:00:26,10166,1,M,NaN,20170317,10226,10142,10166,10216
3,20170317,M3,FI0050674471,SIBXH7J7,08:00:26,"41,5",1,R,NaN,NaN,NaN,NaN,NaN,NaN
4,20170317,M3,FI0050674472,FIBXH7,08:00:26,10169,1,M,NaN,20170317,10226,10142,10166,10216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15647080,20220628,M3,007647891,FIBXN2,19:59:28.277066,8188,1,M,NaN,20220715,8298,8168,8168,8189
15647081,20220628,M3,007647892,FIBXN2,19:59:44.393052,8188,1,M,NaN,20220715,8298,8168,8168,8189
15647082,20220628,M3,007647893,FIBXN2,19:59:47.739983,8186,1,M,NaN,20220715,8298,8168,8168,8189
15647083,20220628,M3,007647894,FIBXN2,19:59:54.692627,8190,1,M,NaN,20220715,8298,8168,8168,8189


In [ ]:
TRADE_IBEX_DATABASE[TRADE_IBEX_DATABASE["StrikePrice"].isna()]

,SessionDate,MarketCode,TradeExecID,ContractCode,ExecTime,TradePrice,Quantity,TradeType,StrikePrice,MaturityDate,HighPrice,LowPrice,FirstPrice,LastPrice
0,20170317,M3,FI0050674468,FIBXH7,08:00:26,10166,1,M,NaN,20170317,10226,10142,10166,10216
1,20170317,M3,FI0050674469,FIBXJ7,08:00:26,10129,1,M,NaN,20170421,10231,10101,10129,10224
2,20170317,M3,FI0050674470,FIBXH7,08:00:26,10166,1,M,NaN,20170317,10226,10142,10166,10216
3,20170317,M3,FI0050674471,SIBXH7J7,08:00:26,"41,5",1,R,NaN,NaN,NaN,NaN,NaN,NaN
4,20170317,M3,FI0050674472,FIBXH7,08:00:26,10169,1,M,NaN,20170317,10226,10142,10166,10216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15647080,20220628,M3,007647891,FIBXN2,19:59:28.277066,8188,1,M,NaN,20220715,8298,8168,8168,8189
15647081,20220628,M3,007647892,FIBXN2,19:59:44.393052,8188,1,M,NaN,20220715,8298,8168,8168,8189
15647082,20220628,M3,007647893,FIBXN2,19:59:47.739983,8186,1,M,NaN,20220715,8298,8168,8168,8189
15647083,20220628,M3,007647894,FIBXN2,19:59:54.692627,8190,1,M,NaN,20220715,8298,8168,8168,8189


Estos trades no tienen ni strike ni maturity. Ver qué hacemos con ello.

In [6]:
print(TRADE_IBEX_DATABASE)

         SessionDate MarketCode   TradeExecID ContractCode         ExecTime  \
0           20170317         M3  FI0050674468       FIBXH7         08:00:26   
1           20170317         M3  FI0050674469       FIBXJ7         08:00:26   
2           20170317         M3  FI0050674470       FIBXH7         08:00:26   
3           20170317         M3  FI0050674471     SIBXH7J7         08:00:26   
4           20170317         M3  FI0050674472       FIBXH7         08:00:26   
...              ...        ...           ...          ...              ...   
15647080    20220628         M3     007647891       FIBXN2  19:59:28.277066   
15647081    20220628         M3     007647892       FIBXN2  19:59:44.393052   
15647082    20220628         M3     007647893       FIBXN2  19:59:47.739983   
15647083    20220628         M3     007647894       FIBXN2  19:59:54.692627   
15647084    20220628         M3     007647895       FIBXN2  19:59:57.914100   

         TradePrice Quantity TradeType StrikePrice 

----------------------------

# Construcción OPTIONS_TRADE_IBEX_DATABASE

In [ ]:
# Read raw_data
TRADE_IBEX_DATABASE_df = pd.read_csv("raw_data/TRADE_IBEX_DATABASE.csv")

# Create OPTIONS_TRADE_IBEX_DATABASE
OPTIONS_TRADE_IBEX_DATABASE = TRADE_IBEX_DATABASE_df.loc[:,[
    "ContractCode",
    "SessionDate",
    "MarketCode",
    "TradeExecID",
    "ExecTime",
    "TradePrice",
    "Quantity",
    "TradeType",
    "StrikePrice",
    "MaturityDate"
]]

# Only options of IBEX
filter_options_IBX = OPTIONS_TRADE_IBEX_DATABASE["ContractCode"].str.startswith(
    ("CIBX", "PIBX"),
    na=False
)
OPTIONS_TRADE_IBEX_DATABASE = OPTIONS_TRADE_IBEX_DATABASE[filter_options_IBX]

# Rename de ContractCode
OPTIONS_TRADE_IBEX_DATABASE = OPTIONS_TRADE_IBEX_DATABASE.rename(columns={"ContractCode":"OptionContractCode"})

# Save CSV
output_file = Path("raw_data/OPTIONS_TRADE_IBEX_DATABASE.csv")
OPTIONS_TRADE_IBEX_DATABASE.to_csv(output_file, index=False, encoding="utf-8")
print(f"Archivo final guardado en: {output_file}")

display(OPTIONS_TRADE_IBEX_DATABASE)

C:\Users\danir\AppData\Local\Temp\ipykernel_9372\2361308103.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  TRADE_IBEX_DATABASE_df = pd.read_csv('raw_data/TRADE_IBEX_DATABASE.csv')


Archivo final guardado en: raw_data\OPTIONS_TRADE_IBEX_DATABASE.csv


,OptionContractCode,SessionDate,MarketCode,TradeExecID,ExecTime,TradePrice,Quantity,TradeType,StrikePrice,MaturityDate
533,PIBX 6000U17,20170317,M3,OM0001668996,09:00:24,14,4,M,6000.0,20170915.0
834,PIBX10100H17,20170317,M3,OM0001668997,09:04:05,8,1,M,10100.0,20170317.0
1011,PIBX 8000Z17,20170317,M3,OM0001668998,09:08:04,160,1,M,8000.0,20171215.0
1059,PIBX10200J17,20170317,M3,OM0001668999,09:09:02,220,1,M,10200.0,20170421.0
1060,PIBX10100H17,20170317,M3,OM0001669000,09:09:02,6,1,M,10100.0,20170317.0
...,...,...,...,...,...,...,...,...,...,...
15645193,CIBX 8500N22,20220628,M3,100144011,17:28:12.120873,52,1,M,8500.0,20220715.0
15645195,PIBX 7500N22,20220628,M3,100144012,17:28:21.709970,11,1,M,7500.0,20220715.0
15645305,CIBX 8500N22,20220628,M3,100144013,17:29:39.189409,49,1,M,8500.0,20220715.0
15645339,CIBX 8500N22,20220628,M3,100144014,17:29:58.629091,49,3,M,8500.0,20220715.0


----------------------------

# Construcción FUTURE_TRADE_IBEX_DATABASE

In [ ]:
# Read raw_data
TRADE_IBEX_DATABASE_df = pd.read_csv("raw_data/TRADE_IBEX_DATABASE.csv")

# Create FUTURE_TRADE_IBEX_DATABASE
FUTURE_TRADE_IBEX_DATABASE = TRADE_IBEX_DATABASE_df.loc[:,[
    "ContractCode",
    "SessionDate",
    "MarketCode",
    "TradeExecID",
    "ExecTime",
    "TradePrice",
    "Quantity",
    "TradeType",
    "StrikePrice",
    "MaturityDate"
]]

# Only futures of IBEX
filter_futures_IBX = FUTURE_TRADE_IBEX_DATABASE["ContractCode"].str.startswith(
    ("FIBX"),
    na=False
)
FUTURE_TRADE_IBEX_DATABASE = FUTURE_TRADE_IBEX_DATABASE[filter_futures_IBX]

# Rename de ContractCode
FUTURE_TRADE_IBEX_DATABASE = FUTURE_TRADE_IBEX_DATABASE.rename(columns={"ContractCode":"FutureContractCode"})

# Save CSV
output_file = Path("raw_data/FUTURE_TRADE_IBEX_DATABASE.csv")
FUTURE_TRADE_IBEX_DATABASE.to_csv(output_file, index=False, encoding="utf-8")
print(f"Archivo final guardado en: {output_file}")

display(FUTURE_TRADE_IBEX_DATABASE)

C:\Users\danir\AppData\Local\Temp\ipykernel_9372\1047499592.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  TRADE_IBEX_DATABASE_df = pd.read_csv('raw_data/TRADE_IBEX_DATABASE.csv')


Archivo final guardado en: raw_data\FUTURE_TRADE_IBEX_DATABASE.csv


,FutureContractCode,SessionDate,MarketCode,TradeExecID,ExecTime,TradePrice,Quantity,TradeType,StrikePrice,MaturityDate
0,FIBXH7,20170317,M3,FI0050674468,08:00:26,10166,1,M,NaN,20170317.0
1,FIBXJ7,20170317,M3,FI0050674469,08:00:26,10129,1,M,NaN,20170421.0
2,FIBXH7,20170317,M3,FI0050674470,08:00:26,10166,1,M,NaN,20170317.0
4,FIBXH7,20170317,M3,FI0050674472,08:00:26,10169,1,M,NaN,20170317.0
5,FIBXJ7,20170317,M3,FI0050674473,08:00:26,10127,1,M,NaN,20170421.0
...,...,...,...,...,...,...,...,...,...,...
15647080,FIBXN2,20220628,M3,7647891,19:59:28.277066,8188,1,M,NaN,20220715.0
15647081,FIBXN2,20220628,M3,7647892,19:59:44.393052,8188,1,M,NaN,20220715.0
15647082,FIBXN2,20220628,M3,7647893,19:59:47.739983,8186,1,M,NaN,20220715.0
15647083,FIBXN2,20220628,M3,7647894,19:59:54.692627,8190,1,M,NaN,20220715.0


----------------------------

# Construcción OPTIONS_UNDERLYING_IBEX_DATABASE

In [ ]:
# Unique options contract codes and their maturity dates
options_maturity_df = OPTIONS_TRADE_IBEX_DATABASE.loc[:,["OptionContractCode","MaturityDate"]].drop_duplicates(subset=["OptionContractCode"], keep="first")
print(f"Options maturity DataFrame shape: {options_maturity_df.shape}")

# Unique futures contract codes and their maturity dates
futures_maturity_df = FUTURE_TRADE_IBEX_DATABASE.loc[:,["FutureContractCode","MaturityDate"]].drop_duplicates(subset=["FutureContractCode"], keep="first")
print(f"Futures maturity DataFrame shape: {futures_maturity_df.shape}")

# Merge
OPTIONS_UNDERLYING_IBEX_DATABASE  = options_maturity_df.merge(
    futures_maturity_df,
    how="left",
    on = "MaturityDate"
)

# Order
column_order = ["OptionContractCode", "FutureContractCode", "MaturityDate"]
OPTIONS_UNDERLYING_IBEX_DATABASE = OPTIONS_UNDERLYING_IBEX_DATABASE[column_order]

# Save CSV
output_file = Path("raw_data/OPTIONS_UNDERLYING_IBEX_DATABASE.csv")
OPTIONS_UNDERLYING_IBEX_DATABASE.to_csv(output_file, index=False, encoding="utf-8")
print(f"Archivo final guardado en: {output_file}")

display(OPTIONS_UNDERLYING_IBEX_DATABASE)

Options maturity DataFrame shape: (8527, 2)
Futures maturity DataFrame shape: (70, 2)
Archivo final guardado en: raw_data\OPTIONS_UNDERLYING_IBEX_DATABASE.csv


,OptionContractCode,FutureContractCode,MaturityDate
0,PIBX 6000U17,FIBXU7,20170915.0
1,PIBX10100H17,FIBXH7,20170317.0
2,PIBX 8000Z17,FIBXZ7,20171215.0
3,PIBX10200J17,FIBXJ7,20170421.0
4,CIBX10000J17,FIBXJ7,20170421.0
...,...,...,...
8522,CIBX 8600W4N22,NaN,20220722.0
8523,CIBX 7500H23,NaN,20230317.0
8524,PIBX 7500W4N22,NaN,20220722.0
8525,CIBX 8800W4N22,NaN,20220722.0


Hay alguno con valores missing. ¿Eliminamos esa data?

# Construcción OPTIONS_TRADE_IBEX_DATABASE

In [ ]:
import pandas as pd
from pathlib import Path

# Read raw data
options_df  = pd.read_csv("raw_data/OPTIONS_TRADE_IBEX_DATABASE.csv")
futures_df  = pd.read_csv("raw_data/FUTURE_TRADE_IBEX_DATABASE.csv")
options_underlying_df  = pd.read_csv("raw_data/OPTIONS_UNDERLYING_IBEX_DATABASE.csv")

# Create exec_datetime (SessionDate + ExecTime)
for df in (options_df, futures_df):
    df["SessionDate"] = df["SessionDate"].astype(str)
    df["ExecTime"] = df["ExecTime"].astype(str)

    df["exec_datetime"] = pd.to_datetime(
        df["SessionDate"] + " " + df["ExecTime"],
        format="mixed"
    )

# Join option with its underlying future
options_df = options_df.merge(
    options_underlying_df,
    on="OptionContractCode",
    how="left"
)

# Rename exec_datetime of future
futures_df = futures_df.rename(columns={"exec_datetime": "exec_datetime_future"})

# Order by exec_datetime
options_df = options_df.sort_values("exec_datetime").reset_index(drop=True)
futures_df = futures_df.sort_values("exec_datetime_future").reset_index(drop=True)

# As-of join: Last trade of the underlying FUTURE with exec_datetime <= exec_datetime of the option
OPTIONS_TRADE_IBEX_DATABASE = pd.merge_asof(
    options_df,
    futures_df,
    by="FutureContractCode",
    left_on="exec_datetime",
    right_on="exec_datetime_future",
    direction="backward",
    suffixes=("", "_future")
)

# Rename columns
OPTIONS_TRADE_IBEX_DATABASE = OPTIONS_TRADE_IBEX_DATABASE.rename(columns={
    "TradePrice": "TradePrice_option",
    "TradePrice_future": "underelayingPrice",
    "exec_datetime_future": "underelayingExecTime"
})

# Select final columns
OPTIONS_TRADE_IBEX_DATABASE = OPTIONS_TRADE_IBEX_DATABASE[[
    "OptionContractCode",
    "FutureContractCode",
    "underelayingPrice",
    "SessionDate",
    "MarketCode",
    "TradeExecID",
    "ExecTime",
    "TradePrice_option",
    "Quantity",
    "TradeType",
    "StrikePrice",
    "MaturityDate",
    "exec_datetime",
    "underelayingExecTime"
]]

# Save CSV
output_file = Path("raw_data/OPTIONS_TRADE_IBEX_DATABASE.csv")
OPTIONS_TRADE_IBEX_DATABASE.to_csv(output_file, index=False, encoding="utf-8")
print(f"Archivo final guardado en: {output_file}")

display(OPTIONS_TRADE_IBEX_DATABASE)

C:\Users\danir\AppData\Local\Temp\ipykernel_29552\2085366734.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  options_df  = pd.read_csv('raw_data/OPTIONS_TRADE_IBEX_DATABASE.csv')
C:\Users\danir\AppData\Local\Temp\ipykernel_29552\2085366734.py:6: DtypeWarning: Columns (3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  futures_df  = pd.read_csv('raw_data/FUTURE_TRADE_IBEX_DATABASE.csv')


Archivo final guardado en: raw_data\OPTIONS_TRADE_IBEX_DATABASE.csv


,OptionContractCode,FutureContractCode,underelayingPrice,SessionDate,MarketCode,TradeExecID,ExecTime,TradePrice_option,Quantity,TradeType,StrikePrice,MaturityDate,exec_datetime,underelayingExecTime
0,PIBX 6000U17,FIBXU7,NaN,20170317,M3,OM0001668996,09:00:24,14,4,M,6000.0,NaN,2017-03-17 09:00:24.000000,NaT
1,PIBX10100H17,FIBXH7,10154,20170317,M3,OM0001668997,09:04:05,8,1,M,10100.0,20170317.0,2017-03-17 09:04:05.000000,2017-03-17 09:03:02.000000
2,PIBX 8000Z17,FIBXZ7,NaN,20170317,M3,OM0001668998,09:08:04,160,1,M,8000.0,NaN,2017-03-17 09:08:04.000000,NaT
3,PIBX10200J17,FIBXJ7,10129,20170317,M3,OM0001668999,09:09:02,220,1,M,10200.0,20170421.0,2017-03-17 09:09:02.000000,2017-03-17 09:09:02.000000
4,PIBX10100H17,FIBXH7,10170,20170317,M3,OM0001669000,09:09:02,6,1,M,10100.0,20170317.0,2017-03-17 09:09:02.000000,2017-03-17 09:09:02.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422867,CIBX 8500N22,FIBXN2,8227,20220628,M3,100144011,17:28:12.120873,52,1,M,8500.0,20220715.0,2022-06-28 17:28:12.120873,2022-06-28 17:28:10.680029
422868,PIBX 7500N22,FIBXN2,8227,20220628,M3,100144012,17:28:21.709970,11,1,M,7500.0,20220715.0,2022-06-28 17:28:21.709970,2022-06-28 17:28:13.814459
422869,CIBX 8500N22,FIBXN2,8225,20220628,M3,100144013,17:29:39.189409,49,1,M,8500.0,20220715.0,2022-06-28 17:29:39.189409,2022-06-28 17:29:39.083192
422870,CIBX 8500N22,FIBXN2,8226,20220628,M3,100144014,17:29:58.629091,49,3,M,8500.0,20220715.0,2022-06-28 17:29:58.629091,2022-06-28 17:29:58.586608
